# Modèle Âge V3 - MobileNetV3Small + CBAM Attention

**Changements par rapport à V2:**

- **Backbone MobileNetV3Small** (au lieu d'EfficientNetB0)
- CBAM Attention (Channel + Spatial) conservé
- Même pipeline de données avec Mixup et rotation
- Fine-tuning en 2 phases

**Avantages de MobileNetV3Small:**

- ~2.5M paramètres (vs ~5.3M pour EfficientNetB0)
- 2-3x plus rapide sur mobile
- TFLite plus petit (~6MB vs ~15MB)


In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
from sklearn.model_selection import train_test_split
from scipy import stats

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks, regularizers
from tensorflow.keras.applications import MobileNetV3Small  # CHANGEMENT
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.losses import Huber

# GPU config
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {gpus}")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (10, 5), "axes.titlesize": 14})

## Configuration


In [ ]:
# Chemins
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data" / "UTKFace"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Hyperparamètres
IMG_SIZE = 128
BATCH_SIZE = 32
WARMUP_EPOCHS = 30
FINETUNE_EPOCHS = 60
INITIAL_LR = 3e-4
FINETUNE_LR = 5e-6

# Huber Loss
HUBER_DELTA = 3.0

# Mixup
MIXUP_ALPHA = 0.2

# Validation
VAL_SPLIT = 0.1
TEST_SPLIT = 0.1

# Buckets d'âge pour stratification
AGE_BINS = [0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 70, 120]
AGE_LABELS = [
    "0-5",
    "6-10",
    "11-15",
    "16-20",
    "21-25",
    "26-30",
    "31-40",
    "41-50",
    "51-60",
    "61-70",
    "70+",
]

print(f"Configuration V3 (MobileNetV3Small):")
print(f"  - Backbone: MobileNetV3Small")
print(f"  - Huber Loss delta: {HUBER_DELTA}")
print(f"  - Mixup alpha: {MIXUP_ALPHA}")
print(f"  - Warmup epochs: {WARMUP_EPOCHS}")
print(f"  - Fine-tune epochs: {FINETUNE_EPOCHS}")

## Chargement des Données


In [ ]:
def parse_utkface_filename(filepath):
    try:
        name = Path(filepath).name.split(".")[0]
        parts = name.split("_")
        if len(parts) < 1 or not parts[0].isdigit():
            return None
        age = int(parts[0])
        if age < 0 or age > 116:
            return None
        return {"filepath": str(filepath), "age": age}
    except Exception:
        return None


# Charger toutes les images
image_paths = list(DATA_DIR.glob("*.jpg*"))
print(f"Images trouvées: {len(image_paths)}")

records = []
for path in tqdm(image_paths, desc="Parsing"):
    record = parse_utkface_filename(path)
    if record:
        records.append(record)

df = pd.DataFrame(records)
print(f"Images valides: {len(df)}")

# Supprimer outliers
z_scores = np.abs(stats.zscore(df["age"]))
df_clean = df[z_scores <= 3].copy()
print(f"Images après nettoyage: {len(df_clean)}")
print(f"Plage d'âge: {df_clean['age'].min()} - {df_clean['age'].max()}")

In [ ]:
# Créer les buckets d'âge
df_clean["age_bucket"] = pd.cut(df_clean["age"], bins=AGE_BINS, labels=AGE_LABELS)

# Distribution
bucket_counts = df_clean["age_bucket"].value_counts().sort_index()
print("Distribution par bucket d'âge:")
print(bucket_counts)

## Split et Oversampling


In [ ]:
# Split stratifié
X = df_clean["filepath"].values
y = df_clean["age"].values

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SPLIT, random_state=SEED, stratify=df_clean["age_bucket"]
)

trainval_buckets = pd.cut(y_trainval, bins=AGE_BINS, labels=AGE_LABELS)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=VAL_SPLIT / (1 - TEST_SPLIT),
    random_state=SEED,
    stratify=trainval_buckets,
)

print(f"Split:")
print(f"  - Train: {len(X_train)}")
print(f"  - Val: {len(X_val)}")
print(f"  - Test: {len(X_test)}")

In [ ]:
def oversample_age_buckets(
    X, y, age_bins, age_labels, target_ratio=2.0, young_boost=None
):
    """Oversampling avec focus sur les jeunes."""
    buckets = pd.cut(y, bins=age_bins, labels=age_labels)
    bucket_counts = pd.Series(buckets).value_counts()
    target_count = int(bucket_counts.median() * target_ratio)

    if young_boost is None:
        young_boost = ["0-5", "6-10", "11-15", "16-20"]

    X_resampled, y_resampled = [], []

    for bucket in age_labels:
        mask = buckets == bucket
        X_bucket = X[mask]
        y_bucket = y[mask]
        current_count = len(X_bucket)

        if current_count == 0:
            continue

        if bucket in young_boost:
            bucket_target = int(target_count * 2.0)
        else:
            bucket_target = target_count

        if current_count < bucket_target:
            repeats = (bucket_target // current_count) + 1
            X_bucket = np.tile(X_bucket, repeats)[:bucket_target]
            y_bucket = np.tile(y_bucket, repeats)[:bucket_target]

        X_resampled.extend(X_bucket)
        y_resampled.extend(y_bucket)

    indices = np.random.permutation(len(X_resampled))
    return np.array(X_resampled)[indices], np.array(y_resampled)[indices]


X_train_balanced, y_train_balanced = oversample_age_buckets(
    X_train, y_train, AGE_BINS, AGE_LABELS
)
print(f"\nAprès oversampling: {len(X_train_balanced)} (était {len(X_train)})")

## Data Pipeline avec Mixup


In [ ]:
# Data augmentation
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.25),  # ±45°
        layers.RandomZoom((-0.2, 0.2)),
        layers.RandomTranslation(0.15, 0.15),
        layers.RandomBrightness(0.25),
        layers.RandomContrast(0.25),
    ],
    name="data_augmentation",
)


def load_and_preprocess(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    label = tf.cast(label, tf.float32)
    return img, label


def mixup_regression(images, labels, alpha=0.2):
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform([], 0.5, 1.0)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam * images + (1 - lam) * tf.gather(images, indices)
    mixed_labels = lam * labels + (1 - lam) * tf.gather(labels, indices)
    return mixed_images, mixed_labels


def create_dataset(
    filepaths, labels, batch_size, shuffle=True, augment=False, use_mixup=False
):
    labels = labels.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))

    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(filepaths), seed=SEED, reshuffle_each_iteration=True
        )

    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)

    if augment:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE,
        )

    if use_mixup:
        ds = ds.map(
            lambda x, y: mixup_regression(x, y, MIXUP_ALPHA),
            num_parallel_calls=tf.data.AUTOTUNE,
        )

    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = create_dataset(
    X_train_balanced,
    y_train_balanced,
    BATCH_SIZE,
    shuffle=True,
    augment=True,
    use_mixup=True,
)
val_ds = create_dataset(
    X_val, y_val, BATCH_SIZE, shuffle=False, augment=False, use_mixup=False
)
test_ds = create_dataset(
    X_test, y_test, BATCH_SIZE, shuffle=False, augment=False, use_mixup=False
)

print("Datasets créés avec Mixup!")

## Modèle avec CBAM Attention + MobileNetV3Small


In [ ]:
def squeeze_excitation_block(x, ratio=16, name_prefix="se"):
    """Squeeze-Excitation (Channel) attention block."""
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D(name=f"{name_prefix}_gap")(x)
    se = layers.Dense(filters // ratio, activation="relu", name=f"{name_prefix}_fc1")(
        se
    )
    se = layers.Dense(filters, activation="sigmoid", name=f"{name_prefix}_fc2")(se)
    se = layers.Reshape((1, 1, filters), name=f"{name_prefix}_reshape")(se)
    return layers.Multiply(name=f"{name_prefix}_scale")([x, se])


def spatial_attention_block(x, kernel_size=7, name_prefix="spatial"):
    """Spatial Attention Block - focus sur les zones de rides."""
    avg_pool = keras.ops.mean(x, axis=-1, keepdims=True)
    max_pool = keras.ops.max(x, axis=-1, keepdims=True)
    concat = layers.Concatenate(axis=-1, name=f"{name_prefix}_concat")(
        [avg_pool, max_pool]
    )
    attention = layers.Conv2D(
        1,
        kernel_size=kernel_size,
        padding="same",
        activation="sigmoid",
        name=f"{name_prefix}_conv",
    )(concat)
    return layers.Multiply(name=f"{name_prefix}_scale")([x, attention])


def cbam_block(x, ratio=16, kernel_size=7, name_prefix="cbam"):
    """CBAM: Channel + Spatial Attention."""
    x = squeeze_excitation_block(x, ratio=ratio, name_prefix=f"{name_prefix}_se")
    x = spatial_attention_block(
        x, kernel_size=kernel_size, name_prefix=f"{name_prefix}_spatial"
    )
    return x


def build_age_model_v3(img_size, trainable_backbone=False):
    """
    Modèle V3 avec MobileNetV3Small + CBAM Attention.

    Changement: EfficientNetB0 -> MobileNetV3Small
    """
    inputs = layers.Input(shape=(img_size, img_size, 3), name="input_image")

    # ════════════════════════════════════════════════════════════
    # BACKBONE: MobileNetV3Small
    # ════════════════════════════════════════════════════════════
    backbone = MobileNetV3Small(
        include_top=False,
        weights="imagenet",
        input_shape=(img_size, img_size, 3),
        include_preprocessing=True,
        pooling=None,
    )
    backbone.trainable = trainable_backbone

    x = backbone(inputs)

    # CBAM attention (Channel + Spatial) - pour focus sur zones de rides
    x = cbam_block(x, ratio=16, kernel_size=7, name_prefix="cbam")

    # Double pooling
    gap = layers.GlobalAveragePooling2D(name="gap")(x)
    gmp = layers.GlobalMaxPooling2D(name="gmp")(x)
    x = layers.Concatenate(name="concat_pool")([gap, gmp])

    # Head de régression
    x = layers.Dense(
        512, kernel_regularizer=regularizers.l2(1e-4), name="head_dense_1"
    )(x)
    x = layers.BatchNormalization(name="head_bn_1")(x)
    x = layers.Activation("relu", name="head_relu_1")(x)
    x = layers.Dropout(0.5, name="head_dropout_1")(x)

    x = layers.Dense(
        256, kernel_regularizer=regularizers.l2(1e-4), name="head_dense_2"
    )(x)
    x = layers.BatchNormalization(name="head_bn_2")(x)
    x = layers.Activation("relu", name="head_relu_2")(x)
    x = layers.Dropout(0.4, name="head_dropout_2")(x)

    x = layers.Dense(
        128, kernel_regularizer=regularizers.l2(1e-4), name="head_dense_3"
    )(x)
    x = layers.BatchNormalization(name="head_bn_3")(x)
    x = layers.Activation("relu", name="head_relu_3")(x)
    x = layers.Dropout(0.3, name="head_dropout_3")(x)

    x = layers.Dense(64, kernel_regularizer=regularizers.l2(1e-4), name="head_dense_4")(
        x
    )
    x = layers.BatchNormalization(name="head_bn_4")(x)
    x = layers.Activation("relu", name="head_relu_4")(x)
    x = layers.Dropout(0.2, name="head_dropout_4")(x)

    outputs = layers.Dense(1, activation="linear", name="age_output")(x)

    model = Model(inputs, outputs, name="AgeModel_V3_MobileNet")
    return model, backbone


model, backbone = build_age_model_v3(IMG_SIZE, trainable_backbone=False)

print(f"\n{'='*60}")
print(f"MODÈLE ÂGE V3 - MobileNetV3Small")
print(f"{'='*60}")
print(f"Backbone: MobileNetV3Small ({len(backbone.layers)} couches)")
print(f"Total params: {model.count_params():,}")
model.summary()

## Phase 1: Warmup


In [ ]:
steps_per_epoch = len(X_train_balanced) // BATCH_SIZE
total_warmup_steps = steps_per_epoch * WARMUP_EPOCHS

warmup_lr_schedule = CosineDecay(
    initial_learning_rate=INITIAL_LR,
    decay_steps=total_warmup_steps,
    alpha=0.1,
)

model.compile(
    optimizer=AdamW(learning_rate=warmup_lr_schedule, weight_decay=1e-5),
    loss=Huber(delta=HUBER_DELTA),
    metrics=["mae", "mse"],
)

warmup_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_mae", patience=15, restore_best_weights=True, mode="min", verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath=str(ARTIFACTS_DIR / "age_v3_mobilenet_warmup.keras"),
        monitor="val_mae",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
]

print(f"Phase 1: Warmup ({WARMUP_EPOCHS} epochs, backbone gelé)")
print(f"Loss: Huber (delta={HUBER_DELTA})")

In [ ]:
warmup_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=WARMUP_EPOCHS,
    callbacks=warmup_callbacks,
    verbose=1,
)

## Phase 2: Fine-tuning


In [ ]:
# Dégeler les dernières couches du backbone
# MobileNetV3Small a ~158 couches, on en dégèle 50
backbone.trainable = True

freeze_until = len(backbone.layers) - 50
for layer in backbone.layers[:freeze_until]:
    layer.trainable = False

trainable_count = sum(
    [tf.reduce_prod(w.shape).numpy() for w in model.trainable_weights]
)
print(f"Phase 2: Fine-tuning ({FINETUNE_EPOCHS} epochs)")
print(f"Couches dégelées: {len(backbone.layers) - freeze_until}/{len(backbone.layers)}")
print(f"Paramètres entraînables: {trainable_count:,}")

In [ ]:
total_finetune_steps = steps_per_epoch * FINETUNE_EPOCHS

finetune_lr_schedule = CosineDecay(
    initial_learning_rate=FINETUNE_LR,
    decay_steps=total_finetune_steps,
    alpha=0.01,
)

model.compile(
    optimizer=AdamW(learning_rate=finetune_lr_schedule, weight_decay=1e-4),
    loss=Huber(delta=HUBER_DELTA),
    metrics=["mae", "mse"],
)

finetune_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_mae", patience=20, restore_best_weights=True, mode="min", verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath=str(ARTIFACTS_DIR / "age_v3_mobilenet_best.keras"),
        monitor="val_mae",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
]

In [ ]:
finetune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    callbacks=finetune_callbacks,
    verbose=1,
)

## Évaluation


In [ ]:
best_model = keras.models.load_model(ARTIFACTS_DIR / "age_v3_mobilenet_best.keras")

# Prédictions
y_pred = best_model.predict(test_ds, verbose=1).flatten()
y_true = y_test

# Métriques
mae = np.mean(np.abs(y_true - y_pred))
mse = np.mean((y_true - y_pred) ** 2)
rmse = np.sqrt(mse)

ss_res = np.sum((y_true - y_pred) ** 2)
ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
r2 = 1 - (ss_res / ss_tot)

print(f"\n{'='*60}")
print(f"RÉSULTATS V3 (MobileNetV3Small) SUR LE TEST SET")
print(f"{'='*60}")
print(f"  - MAE: {mae:.2f} années")
print(f"  - RMSE: {rmse:.2f} années")
print(f"  - R² Score: {r2:.4f}")

In [ ]:
# Visualisations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_true, y_pred, alpha=0.3, s=10)
axes[0].plot([0, 100], [0, 100], "r--", linewidth=2, label="Parfait")
axes[0].set_xlabel("Âge réel")
axes[0].set_ylabel("Âge prédit")
axes[0].set_title(f"Prédictions vs Réalité (R² = {r2:.3f})")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribution des erreurs
errors = y_pred - y_true
axes[1].hist(errors, bins=50, edgecolor="black", alpha=0.7, color="steelblue")
axes[1].axvline(0, color="red", linestyle="--", linewidth=2)
axes[1].set_xlabel("Erreur (années)")
axes[1].set_ylabel("Fréquence")
axes[1].set_title(f"Distribution des erreurs (MAE = {mae:.2f})")

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "age_v3_mobilenet_analysis.png", dpi=150)
plt.show()

## Sauvegarde et Conversion TFLite


In [ ]:
# Sauvegarder le modèle final
final_model_path = ARTIFACTS_DIR / "age_v3_mobilenet_final.keras"
best_model.save(final_model_path)
print(f"Modèle sauvegardé: {final_model_path}")

# Sauvegarder les infos
import json

model_info = {
    "version": "3.0",
    "backbone": "MobileNetV3Small",
    "img_size": IMG_SIZE,
    "task": "regression",
    "input_range": [0, 255],
    "age_range": [0, 116],
    "features": [
        "MobileNetV3Small backbone",
        "CBAM Attention (Channel + Spatial)",
        "Rotation agressive (±45°)",
        "Mixup",
        "Huber Loss",
        "AdamW + L2",
    ],
    "metrics": {
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
    },
}

with open(ARTIFACTS_DIR / "age_v3_mobilenet_info.json", "w") as f:
    json.dump(model_info, f, indent=2)
print("Infos du modèle sauvegardées")

In [ ]:
# Conversion TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

tflite_path = ARTIFACTS_DIR / "age_v3_mobilenet.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

tflite_size_mb = tflite_path.stat().st_size / 1024 / 1024
print(f"Modèle TFLite sauvegardé: {tflite_path}")
print(f"Taille: {tflite_size_mb:.2f} MB")

## Résumé Final


In [ ]:
print("=" * 60)
print("RÉSUMÉ DU MODÈLE ÂGE V3 (MobileNetV3Small)")
print("=" * 60)
print(f"\nArchitecture: MobileNetV3Small + CBAM Attention + Dense Head")
print(f"Input size: {IMG_SIZE}x{IMG_SIZE}x3")
print(f"Dataset: {len(df_clean)} images")
print(f"\nPerformances sur le test set:")
print(f"  - MAE: {mae:.2f} années")
print(f"  - RMSE: {rmse:.2f} années")
print(f"  - R² Score: {r2:.4f}")
print(f"\nTaille TFLite: {tflite_size_mb:.2f} MB")
print(f"\nFichiers sauvegardés:")
print(f"  - {final_model_path}")
print(f"  - {tflite_path}")
print("=" * 60)